---
# DAY 3 — Feature Selection & Classification
---

## Step 10 — Reload Feature Stack

Start a fresh notebook or restart the kernel. Reload everything from Blob Storage.

🎯 Reload `feat_arr` and `feat_names` from Blob Storage and `feature_names.json`.

In [3]:
# Paste your helper functions (get_cc, read_blob, write_blob) here
# YOUR CODE HERE
import numpy as np
import rasterio
from rasterio.io import MemoryFile
from azure.storage.blob import BlobServiceClient
import json 


import config

NO_DATA = -9999.0

#Connection string format: `DefaultEndpointsProtocol=https;AccountName=...;AccountKey=...;EndpointSuffix=core.windows.net`
acc_name = config.ACCOUNT_NAME
acc_key = config.ACCOUNT_KEY

conn_str = f"DefaultEndpointsProtocol=https;AccountName={acc_name};AccountKey={acc_key};EndpointSuffix=core.windows.net"


def get_cc():
    """
    Return an Azure Blob container client using credentials from config.
    """
    # Build connection string and return container client
    # YOUR CODE HERE
    service_client = BlobServiceClient.from_connection_string(conn_str)
    container_client = service_client.get_container_client("sentinel2-data")
    return container_client



def read_blob(blob_name):
    """
    Download a raster file from Blob Storage and return it as a numpy array.
    Returns: (array, rasterio_profile)
    """
    # 1. Download the blob bytes using get_cc()
    # 2. Open with MemoryFile and rasterio
    # 3. Read band 1 as float32
    # 4. Return (array, profile)
    # YOUR CODE HERE
    cc = get_cc()
    blob_bytes = cc.download_blob(blob_name).readall()

    with MemoryFile(blob_bytes) as memfile:
        with memfile.open() as src:
            arr = src.read(1).astype("float32")
            profile = src.profile.copy()

    return arr, profile


def write_blob(arr, profile, blob_name):
    """
    Write a numpy array as a float32 GeoTIFF to Blob Storage.
    """
    # 1. Copy and update profile (dtype=float32, count=1, nodata=NO_DATA)
    # 2. Write array to a MemoryFile
    # 3. Upload to Blob Storage
    # 4. Print confirmation message
    # YOUR CODE HERE
    prof = profile.copy()
    prof.update(dtype="float32", count=1, nodata=NO_DATA, driver="GTiff")
    with MemoryFile() as memfile:
        with memfile.open(**prof) as f:
          f.write(arr, 1)
        
        data = bytes(memfile.getbuffer())
    cc = get_cc()
    cc.upload_blob(blob_name, data, overwrite=True)
    print("data uploaded to cloud")
    


# Test: list files in Blob Storage
cc    = get_cc()
blobs = [b.name for b in cc.list_blobs(name_starts_with='raw/')]
print(f'Files in Blob Storage: {len(blobs)}')
#print(blobs)




# Load feature names from feature_names.json
# YOUR CODE HERE
with open("feature_names.json") as f:
    feature_names = json.load(f)


# Reload all composites and rebuild feat_arr
# (same code as Step 9 — you can copy it)

# Commented out in local
#cc = get_cc()
#data = cc.download_blob(f'{me}/feat_arr.npy').readall()
#feat_arr = np.load(io.BytesIO(data))
#print(f'Loaded feat_arr: {feat_arr.shape}')


#print(f'Feature stack: {feat_arr.shape}')

Files in Blob Storage: 242


## Step 11 — Rasterize Training Data

Upload the training shapefile to JupyterLab (drag and drop all 4 files: `.shp`, `.dbf`, `.shx`, `.prj`).

🎯 Load the shapefile, check/fix the CRS, and rasterize the training polygons onto the same grid as your feature stack.

💡 Use `gpd.read_file()` to load, `.to_crs()` to reproject if needed  
💡 Use `rasterio.features.rasterize()` with `out_shape`, `fill`, `transform` from `ref_profile`  
💡 The field with crop/landcover codes is `grp_1_nb`

In [4]:
import geopandas as gpd
from rasterio.features import rasterize as rio_rasterize

IN_SITU_SHP = 'wallonia_insitu_2023.shp'
FIELD_CODE  = 'grp_1_nb'

# 1. Load the shapefile
# YOUR CODE HERE

# 2. Print: number of polygons, CRS, unique crop codes
# YOUR CODE HERE

# 3. Check CRS matches raster — reproject if not
# YOUR CODE HERE

# 4. Rasterize: burn FIELD_CODE values onto a grid matching feat_arr
# Use fill=int(NO_DATA) for pixels outside polygons
# YOUR CODE HERE

# 5. Print: number of training pixels and number of classes
# YOUR CODE HERE

# ✅ Expected:
# Training pixels: ~35,000+
# Classes: 15–25 unique crop/landcover types

## Step 12 — Build X and y, Train/Test Split

🎯 Extract the feature matrix X and label vector y from pixels that have training labels.

💡 Boolean mask: `mask = cal_arr != int(NO_DATA)`  
💡 `X = feat_arr[mask, :]` — shape (n_samples, n_features)  
💡 Use `train_test_split` with `stratify=y` to ensure all classes appear in both sets

In [5]:
from sklearn.model_selection import train_test_split

# 1. Create boolean mask of labeled pixels
# YOUR CODE HERE

# 2. Extract X (features) and y (labels)
# YOUR CODE HERE

# 3. Replace NaN in X with NO_DATA
# YOUR CODE HERE

# 4. Stratified 80/20 train/test split
# YOUR CODE HERE

# 5. Print shapes
# YOUR CODE HERE

# ✅ Expected:
# X shape: (n_samples, n_features)
# Training: ~28,000  |  Test: ~7,000

## Step 13 — Feature Selection

With up to 156 features, not all are equally useful. Use a quick Random Forest to rank them.

🎯 Train a fast Random Forest on a 10,000-pixel subsample and rank features by Gini importance.

**Group discussion:** After plotting the importances, answer these questions before selecting your subset:
- Which months are most important? Does this make agronomic sense for Wallonia?
- Which bands appear most often in the top features?
- Are the spectral indices (NDVI, NDWI, NDBI) useful?
- How many features do you need before importance drops off significantly?

💡 Use `n_estimators=50` for speed  
💡 `rf.feature_importances_` gives the Gini importance of each feature

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

# 1. Subsample 10,000 training pixels randomly
# YOUR CODE HERE

# 2. Train a quick Random Forest (50 trees) on the subsample
# YOUR CODE HERE

# 3. Build a DataFrame of feature importances, sorted descending
# YOUR CODE HERE

# 4. Print top 15 features
# YOUR CODE HERE

# 5. Plot top 30 as a bar chart, save as feature_importance.png
# YOUR CODE HERE

# ✅ Expected: bar chart showing top 30 features by importance

In [7]:
# FEATURE SELECTION — discuss with your group before running this cell

# How many features will you keep? Justify your choice:
# (Consider: accuracy vs training speed, which features matter most)
TOP_N = None  # <-- set this after your group discussion

# Select top N features
# YOUR CODE HERE

# Create subsetted versions of your training/test matrices and the image
# X_tr_sel, X_te_sel, img_flat_sel
# YOUR CODE HERE

print(f'Selected {TOP_N} features for training.')
print(f'Top features: {selected[:10]}')

Selected None features for training.


NameError: name 'selected' is not defined

## Step 14 — Train the Random Forest

🎯 Train a full Random Forest classifier on the selected features.

💡 Use `n_estimators=100`, `oob_score=True`, `n_jobs=-1` (uses all CPU cores)  
💡 OOB (Out-of-Bag) score is a free accuracy estimate on data not seen by each tree — a useful sanity check  
💡 Aim for OOB > 80%. If lower, revisit your feature selection or check training data quality.

In [ ]:
import time

# Train the Random Forest
# Print training time and OOB accuracy
# YOUR CODE HERE

# ✅ Expected:
# Training time: 30–120s depending on n_features and n_samples
# OOB accuracy: ideally > 80%

## Step 15 — Predict the Full Image

🎯 Apply the trained classifier to every pixel in the image.

💡 The image needs to be reshaped from `(rows, cols, n_features)` to `(rows*cols, n_features)` for prediction  
💡 After prediction, reshape back to `(rows, cols)` to get a 2D classification map  
💡 Replace NaN values with NO_DATA before predicting

In [ ]:
# 1. Reshape image and replace NaN
# YOUR CODE HERE

# 2. Predict (time this — it should take 20–60s on DS3_v2)
# YOUR CODE HERE

# 3. Reshape prediction back to (rows, cols)
# YOUR CODE HERE

# 4. Save to Blob Storage at results/classification_RF.tif
# YOUR CODE HERE

# ✅ Expected:
# Prediction complete in ~30s
# pred_map shape: (rows, cols)
# Unique predicted classes: [3, 21, 69, ...]

---
# DAY 4 — Evaluation & Post-Processing
---

## Step 16 — Accuracy Assessment

🎯 Evaluate your classifier on the held-out test set (20% of training pixels not seen during training).

Report:
- Overall Accuracy (OA)
- Per-class Precision, Recall, F1-score

**Reflection questions for your group:**
- Which classes have the lowest F1-score? Why might this be?
- How does OA compare to the OOB score from training?
- If a class has high precision but low recall, what does that mean?

💡 `classification_report(y_te, y_pred)` gives all per-class metrics in one call

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Predict on test set and compute accuracy
# YOUR CODE HERE

# ✅ Expected: Overall Accuracy + per-class report table

## Step 17 — Reclassify with Crop Dictionary

The classification map uses detailed numeric codes. The crop dictionary maps these to broader, more readable crop groups.

🎯 Load `crop_dictionary.csv` and remap the detailed codes to broader group codes.

💡 Columns: `grp_1_nb` (detailed code) → `grp_A_nb` (broader code) → `grp_A` (name)  
💡 Loop through the LUT rows and replace values in a copy of `pred_map`

In [ ]:
# 1. Load crop_dictionary.csv (upload to JupyterLab first)
# YOUR CODE HERE

# 2. Load classification map from Blob Storage
# YOUR CODE HERE

# 3. Remap detailed codes to broad group codes using the LUT
# YOUR CODE HERE

# 4. Save reclassified map to Blob Storage
# YOUR CODE HERE

# ✅ Expected: reclassification_RF.tif saved to results/ in Blob Storage

## Step 18 — Majority Filter

Classification maps often have 'salt and pepper' noise — isolated pixels classified differently from their neighbours. A majority filter replaces each pixel with the most common class in its neighbourhood.

🎯 Apply a 3×3 majority (mode) filter to smooth the reclassified map.

💡 Use `scipy.ndimage.generic_filter()` with a custom mode function  
💡 `scipy.stats.mode(x.astype(int), keepdims=False).mode` returns the most common value  
💡 Use `mode='nearest'` to handle edges

In [ ]:
from scipy.ndimage import generic_filter
from scipy.stats import mode as sp_mode

def majority_filter(arr, ws=3):
    """
    Apply a majority (mode) filter to a 2D integer array.
    Each pixel is replaced by the most common value in its ws x ws neighbourhood.
    """
    # YOUR CODE HERE
    pass


# Apply filter and save result
print('Applying majority filter...')
# YOUR CODE HERE

# ✅ Expected: reclassification_RF_filtered.tif saved to Blob Storage

## Step 19 — Visualise the Final Map

🎯 Create a publication-quality visualisation of your final classified map with a legend showing crop/land cover names.

💡 Use `matplotlib.colors.ListedColormap` with `plt.cm.tab20` to create a categorical colour map  
💡 Use `matplotlib.patches.Patch` to create legend entries  
💡 Map class codes to names using `code_to_name = dict(zip(lut_df[FIELD_NEW], lut_df[FIELD_NM]))`  
💡 Replace NO_DATA pixels with `np.nan` before plotting so they appear transparent

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

# 1. Get unique class codes from filtered_map
# YOUR CODE HERE

# 2. Build code → name mapping from LUT
# YOUR CODE HERE

# 3. Create colour map
# YOUR CODE HERE

# 4. Plot the map with legend
# YOUR CODE HERE

# 5. Save as final_crop_map.png at dpi=200
# YOUR CODE HERE

# ✅ Expected: colourful crop map with legend showing crop/landcover names

## Step 20 — Download Results

🎯 Download your final GeoTIFF results to the JupyterLab local filesystem so you can save them to your laptop.

Then right-click each file in the JupyterLab file browser and select **Download** to save to your local machine.

In [ ]:
def download_blob(blob_name, local_path):
    """
    Download a file from Blob Storage to the local filesystem.
    """
    # YOUR CODE HERE
    pass


download_blob('results/reclassification_RF_filtered.tif', 'final_map.tif')
download_blob('results/classification_RF.tif',            'raw_classification.tif')
print('Results downloaded. Right-click files in the JupyterLab browser to save locally.')

## Step 21 — Clean Up Azure Resources

⚠️ **Do this only after downloading all results you want to keep.**

1. Go to **portal.azure.com**
2. **Resource groups** → your group resource group
3. Click **Delete resource group**
4. Type the resource group name to confirm → **Delete**

This removes all resources (compute instance, storage, ML workspace) and stops all costs permanently.

Your Azure for Students account and remaining $100 credit are **not affected**.

---

## 🎉 Congratulations!

You have built a complete end-to-end satellite image classification pipeline on Azure:

- ✅ Provisioned cloud infrastructure from scratch
- ✅ Downloaded Sentinel-2 multi-band data via the Copernicus S3 API
- ✅ Applied cloud masking and computed monthly composites
- ✅ Built a multi-temporal feature stack with spectral indices
- ✅ Selected informative features using Random Forest importance
- ✅ Trained and evaluated a crop type classifier
- ✅ Produced a final classified land cover map of Wallonia

---